In [ ]:
# Colab bootstrap: Daten automatisch laden
from pathlib import Path
import os, sys, zipfile, urllib.request, subprocess

GITHUB_USER = "Facuriy"
GITHUB_REPO = "agro-ai-kurs"
BRANCH = "main"
BLOCK_DIR = Path("02_pflanzenzaehlung_computer_vision_DE")
ZIP_NAME = "02_pflanzenzaehlung_computer_vision_DE.zip"
ZIP_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/raw/{BRANCH}/{ZIP_NAME}"

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    # Falls in Colab Pakete fehlen, kann die folgende Zeile aktiviert werden:
    # subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy pandas matplotlib pillow scikit-image"], check=False)
    if not BLOCK_DIR.exists():
        print("Lade Unterrichtsdaten:", ZIP_URL)
        urllib.request.urlretrieve(ZIP_URL, ZIP_NAME)
        with zipfile.ZipFile(ZIP_NAME, "r") as zf:
            zf.extractall(".")
    os.chdir(BLOCK_DIR)
else:
    if Path.cwd().name != BLOCK_DIR.name and BLOCK_DIR.exists():
        os.chdir(BLOCK_DIR)

print("Arbeitsordner:", Path.cwd())
print("Daten bereit:", [p.name for p in Path.cwd().iterdir() if p.is_dir()])

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage import exposure, filters, measure, morphology

DATA_DIR = Path("class_data_plants")
OUT_DIR = Path("class_outputs_plants")
OUT_DIR.mkdir(exist_ok=True)

examples_path = DATA_DIR / "examples.csv"
if not examples_path.exists():
    raise FileNotFoundError(f"Datei fehlt: {examples_path}")

examples = pd.read_csv(examples_path)
examples

In [ ]:
def load_rgb(example_row):
    path = DATA_DIR / example_row["image_file"]
    if not path.exists():
        raise FileNotFoundError(path)
    return np.array(Image.open(path).convert("RGB"))


fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 4))
for ax, (_, row) in zip(axes, examples.iterrows()):
    img = load_rgb(row)
    ax.imshow(img)
    ax.set_title(f"{row['example_id']}\n{row['sensor']}\n{row['difficulty']}")
    ax.axis("off")
plt.tight_layout()

In [ ]:
def excess_green(rgb):
    arr = rgb.astype("float32") / 255.0
    r, g, b = arr[..., 0], arr[..., 1], arr[..., 2]
    exg = 2 * g - r - b
    return exposure.rescale_intensity(exg, in_range="image", out_range=(0, 1))


row = examples.iloc[0]
img = load_rgb(row)
exg = excess_green(img)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title("RGB")
axes[1].imshow(exg, cmap="gray")
axes[1].set_title("Excess Green")
axes[2].hist(exg.ravel(), bins=50, color="green")
axes[2].set_title("Histogramm")
for ax in axes[:2]:
    ax.axis("off")
plt.tight_layout()

In [ ]:
def detect_plants(rgb, threshold=0.08, min_area_px=18, max_area_px=900, smoothing=1.0):
    exg = excess_green(rgb)
    smooth = filters.gaussian(exg, sigma=smoothing)
    mask = smooth > threshold
    mask = morphology.remove_small_objects(mask, min_size=max(1, int(min_area_px)))
    mask = morphology.remove_small_holes(mask, area_threshold=31)
    mask = morphology.opening(mask, morphology.disk(1))

    labels = measure.label(mask)
    rows = []
    keep = np.zeros_like(mask, dtype=bool)
    for region in measure.regionprops(labels):
        if min_area_px <= region.area <= max_area_px:
            y, x = region.centroid
            rows.append({"x": x, "y": y, "area_px": region.area})
            keep[labels == region.label] = True
    return keep, pd.DataFrame(rows)

In [ ]:
# HIER ÄNDERN
EXAMPLE_ID = examples.iloc[0]["example_id"]
THRESHOLD = float(examples.loc[examples.example_id == EXAMPLE_ID, "suggested_threshold"].iloc[0])
MIN_AREA_PX = int(examples.loc[examples.example_id == EXAMPLE_ID, "min_area_px"].iloc[0])
MAX_AREA_PX = int(examples.loc[examples.example_id == EXAMPLE_ID, "max_area_px"].iloc[0])
SMOOTHING = 1.0

print(EXAMPLE_ID, THRESHOLD, MIN_AREA_PX, MAX_AREA_PX)

In [ ]:
row = examples.loc[examples.example_id == EXAMPLE_ID].iloc[0]
img = load_rgb(row)
mask, pred = detect_plants(img, THRESHOLD, MIN_AREA_PX, MAX_AREA_PX, SMOOTHING)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img)
axes[0].set_title("Bild")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Pflanzenmaske")
axes[2].imshow(img)
if len(pred):
    axes[2].scatter(pred.x, pred.y, s=22, c="yellow", marker="+")
axes[2].set_title(f"Detektionen: {len(pred)}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

In [ ]:
def load_ground_truth(example_row):
    path = DATA_DIR / example_row["ground_truth_file"]
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def detection_metrics(pred, gt, tolerance_px=18):
    pred_xy = pred[["x", "y"]].to_numpy(float) if len(pred) else np.empty((0, 2))
    gt_xy = gt[["x", "y"]].to_numpy(float) if len(gt) else np.empty((0, 2))
    used = set()
    tp = 0
    for gx, gy in gt_xy:
        if len(pred_xy) == 0:
            continue
        d = np.sqrt((pred_xy[:, 0] - gx) ** 2 + (pred_xy[:, 1] - gy) ** 2)
        for idx in np.argsort(d):
            if idx not in used and d[idx] <= tolerance_px:
                used.add(int(idx))
                tp += 1
                break
    fp = len(pred_xy) - tp
    fn = len(gt_xy) - tp
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return pd.Series({
        "Referenz": len(gt_xy), "Detektionen": len(pred_xy),
        "TP": tp, "FP": fp, "FN": fn,
        "Precision": precision, "Recall": recall, "F1": f1,
        "Zählfehler": len(pred_xy) - len(gt_xy),
    })


gt = load_ground_truth(row)
metrics = detection_metrics(pred, gt)
metrics

In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(img)
plt.scatter(gt.x, gt.y, s=44, facecolors="none", edgecolors="white", label="Referenz")
if len(pred):
    plt.scatter(pred.x, pred.y, s=24, c="yellow", marker="+", label="Detektion")
plt.title(f"{EXAMPLE_ID}: Referenz vs. Detektion")
plt.axis("off")
plt.legend()
plt.tight_layout()

In [ ]:
results = []
for _, row in examples.iterrows():
    img = load_rgb(row)
    gt = load_ground_truth(row)
    mask, pred = detect_plants(
        img,
        threshold=float(row.suggested_threshold),
        min_area_px=int(row.min_area_px),
        max_area_px=int(row.max_area_px),
        smoothing=1.0,
    )
    m = detection_metrics(pred, gt)
    m["example_id"] = row.example_id
    m["sensor"] = row.sensor
    m["difficulty"] = row.difficulty
    results.append(m)

results = pd.DataFrame(results)
cols = ["example_id", "sensor", "difficulty", "Referenz", "Detektionen", "Precision", "Recall", "F1", "Zählfehler"]
results[cols]

In [ ]:
ax = results.set_index("example_id")[["Precision", "Recall", "F1"]].plot(kind="bar", figsize=(10, 4))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Wert")
ax.set_title("Vergleich der Metriken")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()